### Global Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

### Global Constants

**SI units throughout**

The standard gravitational parameter $\mu$ is a physical constant that represents the strength of a celestial body's gravitational pull, defined as $\mu = GM$, where $G$ is the universal gravitational constant and $M$ is the mass of the celestial body.


In [ ]:
G = 6.674_30e-11
M_EARTH = 5.972_20e24
R_EARTH = 6.371_00e6
MU_EARTH = G * M_EARTH

### Orbital Motion

For a satellite moving at constant speed along a circular path of orbital period $T$, the angular speed $\omega$ is constant and given by $\omega = \frac{2\pi}{T}$. The gravity of the celestial body provides the centripetal force required to keep the satellite on its circular path, so $\frac{GMm}{r^2} = m\omega^2 r$. Solving for $\omega$ gives $\omega = \sqrt{\dfrac{\mu}{r^3}}$.

The equations of motion on the plane ($x_3=0$):
$$
\mathbf{x}(t) = r \begin{bmatrix}\cos(\omega t)\\ \sin(\omega t)\\ 0\end{bmatrix}, \quad
\mathbf{v}(t) = r\omega \begin{bmatrix}-\sin(\omega t)\\ \cos(\omega t)\\ 0\end{bmatrix}, \quad
\mathbf{a}(t) = -\omega^2 \mathbf{x}(t).
$$

In [ ]:
def angular_speed(r: float, mu: float) -> float:
    return np.sqrt(mu/r**3)

def position(t: np.ndarray, r: float, omega: float) -> np.ndarray:
    x1 = r * np.cos(omega * t)
    x2 = r * np.sin(omega * t)
    x3 = np.zeros_like(t)

    return np.stack((x1, x2, x3), axis=-1)

def velocity(t: np.ndarray, r: float, omega: float) -> np.ndarray:
    v1 = -r * omega * np.sin(omega * t)
    v2 =  r * omega * np.cos(omega * t)
    v3 = np.zeros_like(t)

    return np.stack((v1, v2, v3), axis=-1)

def acceleration(x: np.ndarray, omega: float) -> np.ndarray:
    return -omega**2 * x


### Rotational System

Our circular path lives on a 2D-plane, which can be rotated to give configurations in 3D-geometric space. The rotational matrices are:
$$
\mathbf{R_x}(\alpha) = \begin{bmatrix}
    1 & 0 & 0 \\ 
    0 & \cos(\alpha) & -\sin(\alpha) \\ 
    0 & \sin(\alpha) &  \cos(\alpha)
\end{bmatrix}, \quad

\mathbf{R_y}(\beta)  = \begin{bmatrix}
     \cos(\beta) & 0 & \sin(\beta) \\ 
    0 & 1 & 0 \\ 
    -\sin(\beta) & 0 & \cos(\beta)
\end{bmatrix}, \quad

\mathbf{R_z}(\gamma) = \begin{bmatrix}
    \cos(\gamma) & -\sin(\gamma) & 0 \\ 
    \sin(\gamma) &  \cos(\gamma) & 0 \\ 
    0 & 0 & 1
\end{bmatrix}.
$$

Any two combinations of the rotation matrices $\mathbf{x'} = \mathbf{R_x}(\alpha) \mathbf{R_z}(\gamma) \mathbf{x}$, permits all orientations of the plane.



In [ ]:
def rotation_x(alpha: float) -> np.ndarray:
    return np.array([
        [1, 0, 0],
        [0, np.cos(alpha), -np.sin(alpha)],
        [0, np.sin(alpha),  np.cos(alpha)]
    ])

def rotation_y(beta: float) -> np.ndarray:
    return np.array([
        [ np.cos(beta), 0, np.sin(beta)],
        [ 0, 1, 0],
        [-np.sin(beta), 0, np.cos(beta)]
    ])

def rotation_z(gamma: float) -> np.ndarray:
    return np.array([
        [np.cos(gamma), -np.sin(gamma), 0],
        [np.sin(gamma),  np.cos(gamma), 0],
        [0, 0, 1]
    ])


### Some Orbits

- International Space Station Orbit: 400km, 51.6° inclination
- Polar Orbit: 600km, 90° inclination such that it passes over both poles every orbit
- Sun Synchronous Orbit: 600km, ~98.2° slightly retrograde, the near-polar
  orbit used by most Earth-observation satellites
- Geostationary: 35,786km, 0° inclination equatorial, period matches Earth's rotation
- Retrograde equatorial: same altitude as ISS but orbiting backwards
  (inclination = 180°)



In [ ]:
@dataclass(frozen=True, slots=True)
class CircularOrbit:
    radius: float
    inclination: float
    raan: float

    @property
    def omega(self) -> float:
        return angular_speed(self.radius, MU_EARTH)

    @property
    def period(self) -> float:
        return 2 * np.pi / self.omega

    def path(self, t: np.ndarray) -> np.ndarray:
        path = position(t, self.radius, self.omega)
        R = (rotation_z(self.raan) @ rotation_x(self.inclination))
        return path @ R.T


orbits: dict[str, CircularOrbit] = {
    "International Space Station Orbit":
        CircularOrbit(R_EARTH + 400e3, np.deg2rad(51.6), np.deg2rad(0.0)),

    "Polar Orbit":
        CircularOrbit(R_EARTH + 600e3, np.deg2rad(90.0), np.deg2rad(0.0)),

    "Sun Synchronous Orbit":
        CircularOrbit(R_EARTH + 600e3, np.deg2rad(98.2), np.deg2rad(0.0)),

    "Geostationary Orbit":
        CircularOrbit(R_EARTH + 35_786e3, np.deg2rad(0.0), np.deg2rad(0.0)),

    "Retrograde Equatorial Orbit":
        CircularOrbit(R_EARTH + 400e3, np.deg2rad(180.0), np.deg2rad(0.0)),
}


### Render Orbits in 3D Plot

In [ ]:
NUMBER_OF_POINTS = 1_000_000
PLOT_ZOOM = 0.175

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
ax.set_proj_type("ortho")

# mock surface of Earth
u, v = np.mgrid[0:2*np.pi:60j, 0:np.pi:30j]
ex = R_EARTH * np.cos(u) * np.sin(v)
ey = R_EARTH * np.sin(u) * np.sin(v)
ez = R_EARTH * np.cos(v)
ax.plot_surface(ex, ey, ez, color="steelblue",
    alpha=0.4, linewidth=0, shade=True
)

# render orbitals
max_radius = 0.0
for name, orbit in orbits.items():
    max_radius = max(max_radius, orbit.radius)
    t = np.linspace(0, orbit.period, NUMBER_OF_POINTS)

    path = orbit.path(t)
    ax.plot(path[:, 0], path[:, 1], path[:, 2], lw=1.5, label=name)

lim = max_radius * PLOT_ZOOM
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_zlim(-lim, lim)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_zlabel("z (m)")

ax.set_box_aspect([1, 1, 1])
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()
